# Data Cleaning

## 1. Prices data

In [18]:
# import libraries
import pandas as pd
from pathlib import Path
import warnings

# filter User and Filter Warnings for privacy matters
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)

In [19]:
# define raw dataset path
raw_dataset_path = Path('../dataset/raw/entsoe')

zone = 'IT_NORD'

# read data
prices_data = pd.read_csv(raw_dataset_path / 'prices' / f'{zone}-prices.csv')

prices_data

,Unnamed: 0,0
0,2026-01-01 00:00:00+01:00,111.42
1,2026-01-01 00:15:00+01:00,110.58
2,2026-01-01 00:30:00+01:00,110.58
3,2026-01-01 00:45:00+01:00,107.42
4,2026-01-01 01:00:00+01:00,106.01
...,...,...
2972,2026-01-31 23:00:00+01:00,122.77
2973,2026-01-31 23:15:00+01:00,122.77
2974,2026-01-31 23:30:00+01:00,122.77
2975,2026-01-31 23:45:00+01:00,120.00


In [20]:
# rename columns
prices_data = prices_data.rename(
    columns={
        'Unnamed: 0': 'datetime', 
        '0': f'prices_{zone}'
    }
)

prices_data

,datetime,prices_IT_NORD
0,2026-01-01 00:00:00+01:00,111.42
1,2026-01-01 00:15:00+01:00,110.58
2,2026-01-01 00:30:00+01:00,110.58
3,2026-01-01 00:45:00+01:00,107.42
4,2026-01-01 01:00:00+01:00,106.01
...,...,...
2972,2026-01-31 23:00:00+01:00,122.77
2973,2026-01-31 23:15:00+01:00,122.77
2974,2026-01-31 23:30:00+01:00,122.77
2975,2026-01-31 23:45:00+01:00,120.00


In [21]:
# fast check
print(f'''
----Rows x Cols:----
{prices_data.shape}\n
----First rows:----
{prices_data.head()}\n
----Last rows:----
{prices_data.tail()}
''')


----Rows x Cols:----
(2977, 2)

----First rows:----
                    datetime  prices_IT_NORD
0  2026-01-01 00:00:00+01:00          111.42
1  2026-01-01 00:15:00+01:00          110.58
2  2026-01-01 00:30:00+01:00          110.58
3  2026-01-01 00:45:00+01:00          107.42
4  2026-01-01 01:00:00+01:00          106.01

----Last rows:----
                       datetime  prices_IT_NORD
2972  2026-01-31 23:00:00+01:00          122.77
2973  2026-01-31 23:15:00+01:00          122.77
2974  2026-01-31 23:30:00+01:00          122.77
2975  2026-01-31 23:45:00+01:00          120.00
2976  2026-02-01 00:00:00+01:00          140.11



In [22]:
# check what to clean --> Null values?
prices_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2977 entries, 0 to 2976
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   datetime        2977 non-null   str    
 1   prices_IT_NORD  2977 non-null   float64
dtypes: float64(1), str(1)
memory usage: 46.6 KB


In [23]:
print(f'''
----Date description----
{prices_data['datetime'].describe()}\n
----Prices description----
{prices_data[f'prices_{zone}'].describe()}\n
''')


----Date description----
count                          2977
unique                         2977
top       2026-01-01 00:00:00+01:00
freq                              1
Name: datetime, dtype: object

----Prices description----
count    2977.000000
mean      133.805909
std        24.277683
min        85.260000
25%       117.250000
50%       129.460000
75%       148.780000
max       220.000000
Name: prices_IT_NORD, dtype: float64




In [24]:
# convert datetime
prices_data['datetime'] = pd.to_datetime(
    prices_data['datetime']
)

prices_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2977 entries, 0 to 2976
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype                    
---  ------          --------------  -----                    
 0   datetime        2977 non-null   datetime64[us, UTC+01:00]
 1   prices_IT_NORD  2977 non-null   float64                  
dtypes: datetime64[us, UTC+01:00](1), float64(1)
memory usage: 46.6 KB


In [25]:
prices_data['datetime'].diff().value_counts()

datetime
0 days 00:15:00    2976
Name: count, dtype: int64

In [37]:
prices_data['datetime'].min()

Timestamp('2026-01-01 00:00:00+0100', tz='UTC+01:00')

In [38]:
prices_data['datetime'].max()

Timestamp('2026-02-01 00:00:00+0100', tz='UTC+01:00')

In [29]:
prices_data['datetime'].duplicated().sum()

np.int64(0)

In [39]:
# check timezone
print(prices_data['datetime'].dt.tz)

UTC+01:00


In [41]:
# check number of observations per day
daily_counts = (
    prices_data
    .set_index('datetime')
    .resample('D')
    .size()
)

daily_counts.value_counts().sort_index()

1      1
96    31
Name: count, dtype: int64

In [33]:
# function to get average hourly prices with different MTU data
def to_hourly(df, datetime_col='datetime', value_cols=None):
    df = df.copy()
    df[datetime_col] = pd.to_datetime(df[datetime_col])
    df = (
        df
        .set_index(datetime_col)
        .resample('h')[value_cols]
        .mean()
        .reset_index()
    )
    return df

In [35]:
# function call
prices_hourly = to_hourly(
    prices_data,
    datetime_col='datetime',
    value_cols=[f'prices_{zone}']
)

In [45]:
# Number of observations per hour
hourly_counts = (
    prices_data
    .set_index('datetime')
    .resample('h')['prices_IT_NORD']
    .count()
)

hourly_counts.value_counts().sort_index()

prices_IT_NORD
1      1
4    744
Name: count, dtype: int64

In [36]:
prices_hourly

,datetime,prices_IT_NORD
0,2026-01-01 00:00:00+01:00,110.0000
1,2026-01-01 01:00:00+01:00,109.0000
2,2026-01-01 02:00:00+01:00,108.1875
3,2026-01-01 03:00:00+01:00,105.1000
4,2026-01-01 04:00:00+01:00,98.9200
...,...,...
740,2026-01-31 20:00:00+01:00,142.1000
741,2026-01-31 21:00:00+01:00,129.7400
742,2026-01-31 22:00:00+01:00,125.5050
743,2026-01-31 23:00:00+01:00,122.0775


In [43]:
prices_hourly.info()

<class 'pandas.DataFrame'>
RangeIndex: 745 entries, 0 to 744
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype                    
---  ------          --------------  -----                    
 0   datetime        745 non-null    datetime64[us, UTC+01:00]
 1   prices_IT_NORD  745 non-null    float64                  
dtypes: datetime64[us, UTC+01:00](1), float64(1)
memory usage: 11.8 KB


In [44]:
prices_data[
    prices_data['datetime'].between(
        '2026-01-01 00:00',
        '2026-01-01 01:00'
    )
]

,datetime,prices_IT_NORD
0,2026-01-01 00:00:00+01:00,111.42
1,2026-01-01 00:15:00+01:00,110.58
2,2026-01-01 00:30:00+01:00,110.58
3,2026-01-01 00:45:00+01:00,107.42
4,2026-01-01 01:00:00+01:00,106.01


In [47]:
# export clean dataset
output_path = Path('../dataset/clean/prices')

# create folder if non-existent
output_path.mkdir(parents=True, exist_ok=True)

# save in CSV format
prices_hourly.to_csv(output_path / f'prices_{zone}.csv', index=False)